Task 1:
Temperature

In [2]:
"""Temperature class that supports Celsius, Fahrenheit and Kelvin."""

# Two temperatures closer than this are treated as equal. Floating-point
# numbers are not exact, so we never compare them with a plain ==.
TOLERANCE = 1e-9

# The only units the class accepts.
VALID_UNITS = ("C", "F", "K")


def is_close(a, b):
    """Return True if the numbers a and b differ by less than TOLERANCE."""
    return abs(a - b) <= TOLERANCE


def to_kelvin_from(value, unit):
    """Convert a number in the given unit ("C", "F" or "K") to Kelvin.

    Every comparison and conversion goes through Kelvin, so we only need
    one formula per unit instead of one for every pair of units.
    """
    if unit == "C":
        return value + 273.15
    if unit == "F":
        return (value - 32) * 5 / 9 + 273.15
    return value  # already Kelvin


class Temperature:
    """A temperature with a value and a unit (C, F or K)."""

    def __init__(self, value, unit="C"):
        """Create a temperature.

        Raises ValueError if the unit is not C, F or K (any letter case),
        or if the temperature is below absolute zero (0 K).
        """
        # Check the unit first; .upper() makes "c" and "C" the same.
        if not isinstance(unit, str) or unit.upper() not in VALID_UNITS:
            raise ValueError("Unit must be 'C', 'F' or 'K'.")
        unit = unit.upper()

        # Check the value is a real number (bool is excluded on purpose).
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("Value must be a number.")

        # Absolute zero check: nothing can be colder than 0 K.
        kelvin = to_kelvin_from(value, unit)
        if kelvin < 0 and not is_close(kelvin, 0):
            raise ValueError("Temperature is below absolute zero.")

        # A leading underscore is a Python convention for "private".
        self._value = float(value)
        self._unit = unit

    @property
    def value(self):
        """Return the numeric value (read-only, there is no setter)."""
        return self._value

    @property
    def unit(self):
        """Return the unit letter (read-only, there is no setter)."""
        return self._unit

    def to_kelvin(self):
        """Return this temperature as a float in Kelvin."""
        return to_kelvin_from(self._value, self._unit)

    def to_celsius(self):
        """Return this temperature as a float in Celsius."""
        return self.to_kelvin() - 273.15

    def to_fahrenheit(self):
        """Return this temperature as a float in Fahrenheit."""
        return (self.to_kelvin() - 273.15) * 9 / 5 + 32

    def convert_to(self, target_unit):
        """Return a NEW Temperature in target_unit (case-insensitive).

        Raises ValueError if target_unit is not C, F or K.
        """
        if (not isinstance(target_unit, str)
                or target_unit.upper() not in VALID_UNITS):
            raise ValueError("Unit must be 'C', 'F' or 'K'.")
        target_unit = target_unit.upper()

        if target_unit == "C":
            new_value = self.to_celsius()
        elif target_unit == "F":
            new_value = self.to_fahrenheit()
        else:
            # Tiny rounding errors could give -0.0000000001 K.
            new_value = max(self.to_kelvin(), 0.0)
        return Temperature(new_value, target_unit)

    def __repr__(self):
        """Return an unambiguous text, e.g. Temperature(25.0, 'C')."""
        return f"Temperature({self._value}, '{self._unit}')"

    def __str__(self):
        """Return a readable text with two decimals, e.g. 25.00 °C."""
        if self._unit == "K":
            return f"{self._value:.2f} K"
        return f"{self._value:.2f} °{self._unit}"

    def __eq__(self, other):
        """Return True if both temperatures are equal (any unit)."""
        if not isinstance(other, Temperature):
            return NotImplemented  # lets Python try other options
        return is_close(self.to_kelvin(), other.to_kelvin())

    def __ne__(self, other):
        """Return True if the temperatures are not equal."""
        result = self.__eq__(other)
        if result is NotImplemented:
            return result
        return not result

    def __lt__(self, other):
        """Return True if this temperature is colder than other."""
        if not isinstance(other, Temperature):
            return NotImplemented
        # Smaller AND not "equal within tolerance".
        return (self.to_kelvin() < other.to_kelvin()
                and not is_close(self.to_kelvin(), other.to_kelvin()))

    def __le__(self, other):
        """Return True if this temperature is colder than or equal."""
        if not isinstance(other, Temperature):
            return NotImplemented
        return self < other or self == other

    def __gt__(self, other):
        """Return True if this temperature is warmer than other."""
        if not isinstance(other, Temperature):
            return NotImplemented
        return other < self

    def __ge__(self, other):
        """Return True if this temperature is warmer than or equal."""
        if not isinstance(other, Temperature):
            return NotImplemented
        return other <= self

    # Because equality uses a tolerance, we cannot give a reliable hash.
    __hash__ = None

Test: Temperature

In [3]:
"""pytest tests for the Temperature class. Run with: pytest -v"""

import pytest

from temperature import Temperature

# Reference points as (Celsius, Fahrenheit, Kelvin).
REFERENCE_POINTS = [
    (-273.15, -459.67, 0.0),    # absolute zero
    (-40.0, -40.0, 233.15),     # C and F cross over
    (0.0, 32.0, 273.15),        # water freezes
    (37.0, 98.6, 310.15),       # human body
    (100.0, 212.0, 373.15),     # water boils
]


# ----- 1. Initialisation -----

def test_default_unit_is_celsius():
    """No unit given means Celsius."""
    assert Temperature(25).unit == "C"


@pytest.mark.parametrize("unit", ["c", "C", "f", "F", "k", "K"])
def test_units_are_case_insensitive(unit):
    """Lower and upper case units both work."""
    assert Temperature(10, unit).unit == unit.upper()


def test_value_and_unit_are_read_only():
    """Trying to change value or unit raises AttributeError."""
    temp = Temperature(25, "C")
    with pytest.raises(AttributeError):
        temp.value = 3
    with pytest.raises(AttributeError):
        temp.unit = "K"


# ----- 2. Validation -----

@pytest.mark.parametrize("unit", ["X", "", "Celsius", "CC"])
def test_invalid_unit(unit):
    """Unknown units raise ValueError."""
    with pytest.raises(ValueError):
        Temperature(0, unit)


@pytest.mark.parametrize("value, unit",
                         [(-273.16, "C"), (-459.68, "F"), (-0.01, "K")])
def test_below_absolute_zero(value, unit):
    """Temperatures below 0 K raise ValueError in every unit."""
    with pytest.raises(ValueError):
        Temperature(value, unit)


def test_convert_to_invalid_unit():
    """convert_to rejects unknown units."""
    with pytest.raises(ValueError):
        Temperature(0).convert_to("X")


# ----- 3. Conversions -----

@pytest.mark.parametrize("celsius, fahrenheit, kelvin", REFERENCE_POINTS)
def test_conversions(celsius, fahrenheit, kelvin):
    """Every unit converts correctly at every reference point."""
    for value, unit in ((celsius, "C"), (fahrenheit, "F"), (kelvin, "K")):
        temp = Temperature(value, unit)
        assert temp.to_celsius() == pytest.approx(celsius, abs=1e-9)
        assert temp.to_fahrenheit() == pytest.approx(fahrenheit, abs=1e-9)
        assert temp.to_kelvin() == pytest.approx(kelvin, abs=1e-9)
        assert temp.convert_to("c") == Temperature(celsius, "C")


# ----- 4. Round trip -----

@pytest.mark.parametrize("celsius", [-273.15, -40, 0, 37, 100, 1234.5])
def test_round_trip(celsius):
    """C -> F -> K -> C gives back the original temperature."""
    original = Temperature(celsius, "C")
    back = original.convert_to("F").convert_to("K").convert_to("C")
    assert back == original
    assert back.unit == "C"


# ----- 5. Comparisons -----

def test_equality_across_units():
    """The same temperature in different units is equal."""
    assert Temperature(0, "C") == Temperature(32, "F")
    assert Temperature(0, "C") == Temperature(273.15, "K")
    assert Temperature(0, "C") != Temperature(33, "F")


def test_ordering_across_units():
    """<, <=, > and >= work between different units."""
    cold, warm = Temperature(0, "C"), Temperature(100, "F")
    assert cold < warm
    assert cold <= warm
    assert warm > cold
    assert warm >= cold
    assert not warm < cold
    assert Temperature(0, "C") <= Temperature(32, "F")
    assert Temperature(0, "C") >= Temperature(32, "F")


def test_tolerance():
    """Differences below 1e-9 count as equal, larger ones do not."""
    assert Temperature(0, "C") == Temperature(1e-10, "C")
    assert Temperature(0, "C") != Temperature(1e-6, "C")


def test_comparison_with_other_types():
    """Comparing with a non-Temperature returns NotImplemented."""
    temp = Temperature(0)
    assert temp.__eq__(5) is NotImplemented
    assert temp.__lt__(5) is NotImplemented
    assert temp != 5
    with pytest.raises(TypeError):
        temp < 5


# ----- 6. Formatting -----

def test_str():
    """str() gives two decimals and the unit."""
    assert str(Temperature(25, "C")) == "25.00 °C"
    assert str(Temperature(98.6, "f")) == "98.60 °F"
    assert str(Temperature(300, "K")) == "300.00 K"


def test_repr():
    """repr() looks like the constructor call."""
    assert repr(Temperature(25, "c")) == "Temperature(25.0, 'C')"

In [7]:
!pytest temperature_test.py -v

============================= test session starts ==============================
platform darwin -- Python 3.13.15, pytest-9.2.0, pluggy-1.7.0 -- /Users/kirangiri/Documents/MDS/INF201/Deliverable4/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/kirangiri/Documents/MDS/INF201/Deliverable4/INF230-Deliverable4
plugins: platformdirs-4.14.8
collected 33 items                                                             

temperature_test.py::test_default_unit_is_celsius PASSED                 [  3%]
temperature_test.py::test_units_are_case_insensitive[c] PASSED           [  6%]
temperature_test.py::test_units_are_case_insensitive[C] PASSED           [  9%]
temperature_test.py::test_units_are_case_insensitive[f] PASSED           [ 12%]
temperature_test.py::test_units_are_case_insensitive[F] PASSED           [ 15%]
temperature_test.py::test_units_are_case_insensitive[k] PASSED           [ 18%]
temperature_test.py::test_units_are_case_insensitive[K] PASSED           [ 21%]
temperature_

Task 2: Fraction

In [4]:
"""Fraction class with automatic simplification."""

from math import gcd


class Fraction:
    """A fraction such as 3/4, stored in its simplest form."""

    def __init__(self, numerator, denominator=1):
        """Create a fraction.

        Both parts must be integers and the denominator must be > 0.
        The fraction is simplified, so Fraction(2, 4) becomes 1/2.
        """
        # bool is a subclass of int in Python, so exclude it explicitly.
        for part in (numerator, denominator):
            if isinstance(part, bool) or not isinstance(part, int):
                raise TypeError("Numerator and denominator must be ints.")
        if denominator <= 0:
            raise ValueError("Denominator must be strictly positive.")

        # Divide both parts by their greatest common divisor.
        divisor = gcd(numerator, denominator)
        self.numerator = numerator // divisor
        self.denominator = denominator // divisor

    def __repr__(self):
        """Return an unambiguous text, e.g. Fraction(1, 2)."""
        return f"Fraction({self.numerator}, {self.denominator})"

    def __str__(self):
        """Return a readable text, e.g. 1/2 (or just 2 for 2/1)."""
        if self.denominator == 1:
            return str(self.numerator)
        return f"{self.numerator}/{self.denominator}"

    # ----- Comparisons -----
    # a/b compared with c/d is the same as a*d compared with c*b
    # (valid because both denominators are positive).

    def _difference(self, other):
        """Return self.numerator*other.denominator minus the cross product.

        The result is negative, zero or positive, which tells us whether
        self is smaller than, equal to, or greater than other.
        """
        return (self.numerator * other.denominator
                - other.numerator * self.denominator)

    def __eq__(self, other):
        """Return True if both fractions have the same value."""
        if not isinstance(other, Fraction):
            return NotImplemented
        return self._difference(other) == 0

    def __ne__(self, other):
        """Return True if the fractions have different values."""
        result = self.__eq__(other)
        if result is NotImplemented:
            return result
        return not result

    def __lt__(self, other):
        """Return True if self is smaller than other."""
        if not isinstance(other, Fraction):
            return NotImplemented
        return self._difference(other) < 0

    def __le__(self, other):
        """Return True if self is smaller than or equal to other."""
        if not isinstance(other, Fraction):
            return NotImplemented
        return self._difference(other) <= 0

    def __gt__(self, other):
        """Return True if self is greater than other."""
        if not isinstance(other, Fraction):
            return NotImplemented
        return self._difference(other) > 0

    def __ge__(self, other):
        """Return True if self is greater than or equal to other."""
        if not isinstance(other, Fraction):
            return NotImplemented
        return self._difference(other) >= 0

    def __hash__(self):
        """Return a hash so fractions can be used in sets and dicts."""
        return hash((self.numerator, self.denominator))

    # ----- Other special methods -----

    def __add__(self, other):
        """Return self + other as a new Fraction."""
        if not isinstance(other, Fraction):
            return NotImplemented
        new_num = (self.numerator * other.denominator
                   + other.numerator * self.denominator)
        new_den = self.denominator * other.denominator
        return Fraction(new_num, new_den)

    def __sub__(self, other):
        """Return self - other as a new Fraction."""
        if not isinstance(other, Fraction):
            return NotImplemented
        return self + (-other)

    def __mul__(self, other):
        """Return self * other as a new Fraction."""
        if not isinstance(other, Fraction):
            return NotImplemented
        return Fraction(self.numerator * other.numerator,
                        self.denominator * other.denominator)

    def __truediv__(self, other):
        """Return self / other as a new Fraction.

        Raises ZeroDivisionError when dividing by zero.
        """
        if not isinstance(other, Fraction):
            return NotImplemented
        if other.numerator == 0:
            raise ZeroDivisionError("Cannot divide by zero.")
        new_num = self.numerator * other.denominator
        new_den = self.denominator * other.numerator
        # Keep the denominator positive by moving the sign to the top.
        if new_den < 0:
            new_num, new_den = -new_num, -new_den
        return Fraction(new_num, new_den)

    def __neg__(self):
        """Return the negative of this fraction, e.g. -(1/2) = -1/2."""
        return Fraction(-self.numerator, self.denominator)

    def __abs__(self):
        """Return the absolute value, e.g. abs(-1/2) = 1/2."""
        return Fraction(abs(self.numerator), self.denominator)

    def __float__(self):
        """Return the decimal value, e.g. float(1/4) = 0.25."""
        return self.numerator / self.denominator

Fraction Test

In [5]:
"""pytest tests for the Fraction class. Run with: pytest -v"""

import pytest

from fraction import Fraction


def test_init_simplifies():
    """Fractions are stored in simplest form."""
    f = Fraction(2, 4)
    assert (f.numerator, f.denominator) == (1, 2)
    assert Fraction(-6, 8) == Fraction(-3, 4)


@pytest.mark.parametrize("denominator", [0, -1])
def test_denominator_must_be_positive(denominator):
    """Zero or negative denominators are rejected."""
    with pytest.raises(ValueError):
        Fraction(1, denominator)


@pytest.mark.parametrize("num, den",
                         [(1.5, 2), (1, 2.0), ("1", 2), (True, 2)])
def test_integers_required(num, den):
    """Non-integer parts are rejected."""
    with pytest.raises(TypeError):
        Fraction(num, den)


def test_equality():
    """Equal values are equal, different values are not."""
    assert Fraction(1, 2) == Fraction(2, 4)
    assert Fraction(1, 2) != Fraction(1, 3)
    assert Fraction(1, 2) != "1/2"


def test_ordering():
    """<, <=, > and >= give the right answers."""
    small, big = Fraction(1, 3), Fraction(1, 2)
    assert small < big
    assert small <= big
    assert big > small
    assert big >= small
    assert small <= Fraction(2, 6)
    assert small >= Fraction(2, 6)
    assert not big < small
    assert Fraction(-1, 2) < Fraction(1, 3)


def test_ordering_with_other_type():
    """Comparing with a non-Fraction raises TypeError."""
    with pytest.raises(TypeError):
        Fraction(1, 2) < "x"


def test_add_and_subtract():
    """Addition and subtraction."""
    assert Fraction(1, 2) + Fraction(1, 3) == Fraction(5, 6)
    assert Fraction(1, 2) - Fraction(1, 3) == Fraction(1, 6)


def test_multiply_and_divide():
    """Multiplication and division, including negative and zero."""
    assert Fraction(2, 3) * Fraction(3, 4) == Fraction(1, 2)
    assert Fraction(1, 2) / Fraction(1, 4) == Fraction(2, 1)
    assert Fraction(1, 2) / Fraction(-1, 4) == Fraction(-2, 1)
    with pytest.raises(ZeroDivisionError):
        Fraction(1, 2) / Fraction(0, 5)


def test_neg_abs_float():
    """Negation, absolute value and conversion to float."""
    assert -Fraction(1, 2) == Fraction(-1, 2)
    assert abs(Fraction(-1, 2)) == Fraction(1, 2)
    assert float(Fraction(1, 4)) == 0.25


def test_str_repr_hash():
    """Text output and hashing."""
    assert str(Fraction(3, 6)) == "1/2"
    assert str(Fraction(4, 2)) == "2"
    assert repr(Fraction(3, 6)) == "Fraction(1, 2)"
    assert hash(Fraction(1, 2)) == hash(Fraction(2, 4))

In [8]:
!pytest fraction_test.py -v

============================= test session starts ==============================
platform darwin -- Python 3.13.15, pytest-9.2.0, pluggy-1.7.0 -- /Users/kirangiri/Documents/MDS/INF201/Deliverable4/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/kirangiri/Documents/MDS/INF201/Deliverable4/INF230-Deliverable4
plugins: platformdirs-4.14.8
collected 14 items                                                             

fraction_test.py::test_init_simplifies PASSED                            [  7%]
fraction_test.py::test_denominator_must_be_positive[0] PASSED            [ 14%]
fraction_test.py::test_denominator_must_be_positive[-1] PASSED           [ 21%]
fraction_test.py::test_integers_required[1.5-2] PASSED                   [ 28%]
fraction_test.py::test_integers_required[1-2.0] PASSED                   [ 35%]
fraction_test.py::test_integers_required[1-2] PASSED                     [ 42%]
fraction_test.py::test_integers_required[True-2] PASSED                  [ 50%]
fraction_tes